In [8]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.distributions.empirical_distribution import ECDF 
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

from renewable_data_load import *
from sei_computation import compute_sei, compute_standardized_index, stack_doy_year_to_time

import time

In [16]:
reference_gwl = 0.8
target_gwls = [0.8, 2.0]
domain = 'd02'
variable ='load'

simulations = ["mpi-esm1-2-hr", "miroc6", "taiesm1", "ec-earth3"]

# Data directory
data_dir = Path("../../data/")

# load data

In [9]:
# load and merge data from each region
peak_load_directory = "../../data/demand/DailyPeaks/"
# get list of csv files in directory
import os
csv_files = [f for f in os.listdir(peak_load_directory) if f.endswith('.csv')]

region_datasets = []  # Collect datasets with region names

for file in csv_files:
    region_name = file.split('_')[0]  # assuming the format is REGION_daily_peaks_reference_2025.csv
    df = pd.read_csv(os.path.join(peak_load_directory, file))
    df = df.rename(columns={'Date': 'time'})
    df.set_index('time', inplace=True)
    ds = df.to_xarray()
    
    # Add region name as coordinate
    ds = ds.expand_dims({'region': [region_name]})
    region_datasets.append(ds)

# Concatenate all regions with their names
peak_demand_ds = xr.concat(region_datasets, dim='region')

peak_demand_ds['time'] = pd.to_datetime(peak_demand_ds.time.values)
peak_demand_ds = peak_demand_ds.convert_calendar("noleap")

In [10]:
peak_demand_ds = peak_demand_ds.rename({'EC_EARTH3':'ec-earth3',
                                        'MIROC6':'miroc6',
                                        'MPI_ESM1_2_HR':'mpi-esm1-2-hr',
                                        'TAIESM1':'taiesm1'})

In [17]:
# get reference period of GWL 0.8 for each model

reference_gwl = 0.8
resource = 'demand'
module = 'reference'

for simulation in peak_demand_ds.data_vars:
    peak_demand_da = peak_demand_ds[simulation]
    # Get bounds for reference GWL period
    WRF_sim_name = sim_name_dict[simulation]
    model = WRF_sim_name.split("_")[1]
    ensemble_member = WRF_sim_name.split("_")[2]
    ref_start_year, ref_end_year = get_gwl_crossing_period(model, ensemble_member, reference_gwl)

    peak_demand_ref = peak_demand_da.sel(time=slice(f"{ref_start_year}-01-01",f"{ref_end_year}-12-31"))

    sei = compute_sei(peak_demand_ref, peak_demand_ds[simulation], window_size=60, fill_missing_year=True)
    
    save_plots = True
    if save_plots:
        # Create output directory if it doesn't exist
        output_dir = Path(f"figures/{resource}_{module}")
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # Plot for each region
        for region_name in sei.region.values:
            fig, ax = plt.subplots(figsize=(12, 4))
            xr.plot.imshow(
                sei.sel(region=region_name).T,
                ax=ax,
                levels=[-1.96, -1.64, -1.28, 0, 1.28, 1.64, 1.96],
                cmap="PuOr_r",
                cbar_kwargs={'pad': 0.02}
            )
            plt.title(f"{region_name} {simulation} Daily Generation {resource} SEI")
            plt.ylabel("Year")
            plt.xlabel("Day of Year")
            plt.tight_layout()
            
            # Save figure
            fig_filename = f"{region_name}_{simulation}_{resource}_{module}_SEI.png"
            plt.savefig(output_dir / fig_filename, dpi=300, bbox_inches='tight')
            plt.close()
            
        print(f"Saved SEI plots for all regions to {output_dir}")
    else:
        # Original single region plot
        region = 'PG&E'
        fig, ax = plt.subplots(figsize=(12, 4))
        xr.plot.imshow(
            sei.sel(region=region).T,
            ax=ax,
            levels=[-1.96, -1.64, -1.28, 0, 1.28, 1.64, 1.96],
            cmap="PuOr_r",
            cbar_kwargs={'pad': 0.02}
        )
        plt.title(f"{region} {simulation} Daily Generation {resource} SEI")
        plt.ylabel("Year")
        plt.xlabel("Day of Year")
        plt.tight_layout()
        plt.show()


    sei = stack_doy_year_to_time(sei)
    # save SEI to netCDF in the 
    output_path = f"{data_dir}/SEI/{resource}_{module}_{domain}_{variable}_{simulation}_gwlref{reference_gwl}_regional_SEI.nc"
    sei.to_netcdf(output_path)


    # Create binary drought mask for entire time series
    # 1 where drought_ds < 0 (below threshold), 0 otherwise
    drought_mask = xr.where(sei > 1.28, 1, 0)
    drought_mask.name = "demand_mask"

    # save into the drought mask folder
    drought_mask = drought_mask.load()

    # Save drought mask to file
    mask_output_file = f"{data_dir}/drought_masks/{resource}_{module}_{domain}_{variable}_{simulation}_ts_regional_drought_mask.zarr"

    # Add metadata to drought mask
    drought_mask.attrs['resource'] = resource
    drought_mask.attrs['module'] = module
    drought_mask.attrs['domain'] = domain
    drought_mask.attrs['variable'] = variable
    drought_mask.attrs['simulation'] = simulation
    drought_mask.attrs['reference_gwl'] = float(reference_gwl)
    drought_mask.attrs['description'] = 'Binary demand mask: 1 = high demand (above threshold), 0 = normal demand'

    # Clear encoding from data variable and all coordinates to avoid codec conflicts
    drought_mask.encoding = {}
    for coord in drought_mask.coords:
        drought_mask.coords[coord].encoding = {}

    # Save with Zarr v3 - let xarray choose v3-compatible codecs
    # Only specify dtype, let compression use v3 defaults
    encoding = {'demand_mask': {'dtype': 'int32'}}
    drought_mask.to_zarr(mask_output_file, encoding=encoding, mode='w', consolidated=True)

Saved SEI plots for all regions to figures/demand_reference
Saved SEI plots for all regions to figures/demand_reference
Saved SEI plots for all regions to figures/demand_reference
Saved SEI plots for all regions to figures/demand_reference
